In [ ]:
#  ZEHRA ÇEPNİ - 2021302102
#  FINAL ASSIGNMENT
#  14/08/2026
#  Drive Link for Records: https://drive.google.com/drive/folders/1T7QUFJsdg9LoDejPGjWYjobpIJvE_xvB
#----------------------------------------

#  QUESTION 1 - SEARCH ENGINE v3.0
#  v2.1 -> v3.0: the same URL is not added more than once
#                to the URL list of a given keyword.

import urllib.request


# ---------- 1) Link extraction ----------

def get_next_target(page):
    start_link = page.find('<a href=')
    if start_link == -1:               # no link left, stop here
        return None, 0
    start_quote = page.find('"', start_link)
    end_quote = page.find('"', start_quote + 1)
    url = page[start_quote + 1:end_quote]
    return url, end_quote


def get_all_links(page):
    links = []
    while True:
        url, endpos = get_next_target(page)
        if url:
            links.append(url)
            page = page[endpos:]
        else:
            break
    return links


# ---------- 2) Downloading a page ----------

def getPage(url):
    try:
        page = urllib.request.urlopen(url).read()
        page = page.decode("utf-8")
        return page
    except:
        return ""


# ---------- 3) Merging two lists ----------

def union(p, q):
    for e in q:
        if e not in p:
            p.append(e)


# ---------- 4) Removing HTML tags (from v2.1) ----------

def getClearPage(content):
    result = ""
    while True:
        start_tag = content.find('<')
        if start_tag == -1:
            result = result + content
            break
        result = result + content[:start_tag]
        end_tag = content.find('>', start_tag)
        if end_tag == -1:
            break
        content = content[end_tag + 1:]
    return result


# ---------- 5) Index functions ----------

def addToIndex_v21(index, keyword, url):
    # OLD VERSION, kept only so that we can compare it with v3.0 later.
    # It appends the URL every single time, without checking anything.
    if keyword in index:
        index[keyword].append(url)
    else:
        index[keyword] = [url]


def addToIndex(index, keyword, url):
    # v3.0 VERSION
    # Before appending, we check whether this URL is already stored for
    # this keyword.

    if keyword in index:
        if url not in index[keyword]:
            index[keyword].append(url)
    else:
        index[keyword] = [url]


def addPageToIndex_v21(index, url, content):
    words = content.split()
    for word in words:
        addToIndex_v21(index, word, url)


def addPageToIndex(index, url, content):
    words = content.split()
    for word in words:
        addToIndex(index, word, url)


def lookup(index, keyword):
    if keyword in index:
        return index[keyword]
    return None


# ---------- 6) Crawlers ----------

def crawlWeb_v21(seed):
    # OLD VERSION, used only to produce a comparison index.
    tocrawl = [seed]
    crawled = []
    index = {}
    while tocrawl:
        page = tocrawl.pop()
        if page not in crawled:
            content = getPage(page)
            union(tocrawl, get_all_links(content))
            addPageToIndex_v21(index, page, getClearPage(content))
            crawled.append(page)
    return index


def crawlWeb(seed):
    # v3.0 VERSION
    tocrawl = [seed]
    crawled = []
    index = {}
    while tocrawl:
        page = tocrawl.pop()
        if page not in crawled:
            content = getPage(page)
            # IMPORTANT: links must be taken from the RAW html,
            # before getClearPage removes the tags
            union(tocrawl, get_all_links(content))
            # only the cleaned text goes into the index
            addPageToIndex(index, page, getClearPage(content))
            crawled.append(page)
    return index


# ---------- 7) Part a: run v3.0 ----------

seed = "https://searchengineplaces.com.tr/"
index3 = crawlWeb(seed)
print("Number of keywords in index3:", len(index3))


# ---------- 8) Part b: testing ----------

def countAllUrls(index):
    """
    Adds up the lengths of all URL lists in the index. This is NOT the
    same as len(index): len(index) counts the keywords, while this counts
    how many URL entries are stored in total. The v3.0 change reduces
    this total, but leaves the number of keywords unchanged.
    """
    total = 0
    for keyword in index:
        total = total + len(index[keyword])
    return total


def testNoDuplicates(index):
    """
    For each keyword we rebuild its URL list in a second list called
    'seen'. A URL is copied into 'seen' only if it is not already there,
    so 'seen' can never contain a repetition.

    If the original list turns out to be longer than the rebuilt one,
    some URL was skipped while copying, which can only happen if that
    URL appeared more than once. Comparing the two lengths is therefore
    enough, and we do not have to compare every URL with every other one.

    Only the failing keywords are collected, so the output stays short
    however large the index is.
    """
    problems = []

    for keyword in index:
        urls = index[keyword]
        seen = []                     # URLs already copied for this keyword
        for url in urls:
            if url not in seen:
                seen.append(url)
        if len(urls) != len(seen):    # something was skipped -> duplicate
            problems.append(keyword)

    if problems:
        print("TEST FAILED -", len(problems), "keyword(s) contain duplicate URLs.")
        print("Examples:", problems[:5])
        return False
    else:
        print("TEST PASSED -", len(index), "keywords checked, no duplicates found.")
        return True


# --- Test 1: the v3.0 index must be clean ---
testNoDuplicates(index3)

# --- Test 2: the same test on a v2.1 index, which must FAIL. ---
# Running the test on an index we know is faulty proves that the test
# really detects duplicates, instead of printing PASSED for any input.
index2 = crawlWeb_v21(seed)
testNoDuplicates(index2)

# --- Comparison of the two versions ---
# The keyword counts are expected to be EQUAL: v3.0 does not remove any
# keyword, it only shortens the URL list belonging to each keyword.
# The total number of stored URLs is what must drop.
print("keywords   v2.1:", len(index2), " v3.0:", len(index3))
print("total URLs v2.1:", countAllUrls(index2), " v3.0:", countAllUrls(index3))

Number of keywords in index3: 350
TEST PASSED - 350 keywords checked, no duplicates found.
TEST FAILED - 47 keyword(s) contain duplicate URLs.
Examples: ['Welcome', 'This', 'site', 'was', 'for']
keywords   v2.1: 350  v3.0: 350
total URLs v2.1: 769  v3.0: 585


In [ ]:

#  QUESTION 2 - SEARCH ENGINE v3.1
#  v3.0 -> v3.1: punctuation is stripped from words before they
#                are indexed, so "Ankara" and "Ankara," become
#                one single keyword.


# ---------- Part a: the noPunct procedure ----------

PUNCT = "!,.()?’'"


def noPunct(word):
    """
    Step 1: REMOVE punctuation from the beginning (while loop, because
            several marks can be stacked, e.g. "!!Ankara").
    Step 2: REMOVE punctuation from the end, e.g. "Programming’ve?)".
    Step 3: the outer marks are gone now, so a remaining apostrophe must
            be inside the word ("Python’s"); cut the word there.
    Order matters: if step 3 ran first, "’Python" would lose everything.
    """
    # step 1: leading punctuation
    while word != "" and word[0] in PUNCT:
        word = word[1:]

    # step 2: trailing punctuation
    while word != "" and word[-1] in PUNCT:
        word = word[:-1]

    # step 3: internal apostrophe. We loop over the POSITIONS, because we
    # need the position number i to cut the word with word[:i].
    for i in range(len(word)):
        if word[i] in "’'":
            word = word[:i]
            break                  # the first apostrophe is enough
    return word


# ---------- Part b: testing noPunct ----------

def testNoPunct(testCases):
    """
    Each test case is a pair: the word to process and the keyword we
    expect back, written by hand since we know the correct answer.
    Only the failing cases are printed, so we do not have to read
    through a long list of correct results.
    """
    failed = 0
    for word, expected in testCases:
        result = noPunct(word)
        if result != expected:
            print("FAILED:", word, "-> got '" + result + "', expected '" + expected + "'")
            failed = failed + 1

    if failed == 0:
        print("TEST PASSED -", len(testCases), "words processed correctly.")
    else:
        print("TEST FAILED -", failed, "out of", len(testCases), "words were wrong.")


# first word list given in the question
list1 = [('(Python,', 'Python'),
         ('Python', 'Python'),
         ('(Python)', 'Python'),
         ('Python.', 'Python'),
         ('Python!', 'Python'),
         ('Python,', 'Python'),
         ('Python’s.', 'Python'),
         ('Python’ve.', 'Python')]

# second word list given in the question
list2 = [('Trees’', 'Trees'),
         ('Trees’re.', 'Trees'),
         ('Trees?', 'Trees'),
         ('(Trees’)', 'Trees'),
         ('Trees', 'Trees'),
         ('Programming’s?', 'Programming'),
         ('(Programming’s!', 'Programming'),
         ('Programming', 'Programming'),
         ('Programming’re', 'Programming'),
         ('Programming’ve?)', 'Programming')]

# extra cases of our own, to check that the procedure generalises
list3 = [('(Ankara),', 'Ankara'),
         ('!!Ankara??', 'Ankara'),     # marks stacked on both sides
         ('Ankara', 'Ankara'),         # a clean word stays unchanged
         ('...', ''),                  # only punctuation -> empty string
         ("Ankara's", 'Ankara')]       # ASCII apostrophe

testNoPunct(list1)
testNoPunct(list2)
testNoPunct(list3)


# ---------- Part c: integrating noPunct ----------

def addPageToIndex_v31(index, url, content):
    # v3.1: every word is cleaned before it reaches the index
    words = content.split()
    for word in words:
        keyword = noPunct(word)
        if keyword != "":                     # skip pure-punctuation words
            addToIndex(index, keyword, url)   # addToIndex from v3.0 is reused


def crawlWeb_v31(seed):
    # same as v3.0, only the addPageToIndex call changes
    tocrawl = [seed]
    crawled = []
    index = {}
    while tocrawl:
        page = tocrawl.pop()
        if page not in crawled:
            content = getPage(page)
            union(tocrawl, get_all_links(content))   # links from the RAW html
            addPageToIndex_v31(index, page, getClearPage(content))
            crawled.append(page)
    return index


# index3 from question 1 is left untouched, so the two are comparable
index31 = crawlWeb_v31(seed)

print("Number of keywords in index31 (v3.1):", len(index31))
print("Number of keywords in index3  (v3.0):", len(index3))


# ---------- Testing the integration ----------

def testIndexNoPunct(index):
    """
    Checks the characters of every keyword. If any of them is a
    punctuation mark, that keyword was not cleaned properly. An empty
    keyword counts as a problem too, since it should never be indexed.
    """
    problems = []
    for keyword in index:
        if keyword == "":
            problems.append(keyword)
            continue
        for c in keyword:
            if c in PUNCT:
                problems.append(keyword)
                break              # one bad character is enough
    if problems:
        print("TEST FAILED -", len(problems), "keyword(s) still contain punctuation.")
        print("Examples:", problems[:5])
        return False
    else:
        print("TEST PASSED -", len(index), "keywords checked, none contain punctuation.")
        return True


# the v3.1 index must be clean
testIndexNoPunct(index31)

# the same test on index3, the v3.0 index, which must FAIL: it still holds
# keywords like "Ankara,", so this proves the test really detects punctuation
testIndexNoPunct(index3)

# the property gained in v3.0 must survive in v3.1
testNoDuplicates(index31)


# ---------- Comparing the two versions ----------

# unlike question 1, here the KEYWORD COUNT is the meaningful figure:
# the punctuated forms merge, so index31 must be clearly smaller
print("keywords v3.0:", len(index3), " v3.1:", len(index31))

print("v3.0 'Ankara'  ->", lookup(index3, "Ankara"))
print("v3.0 'Ankara,' ->", lookup(index3, "Ankara,"))
print("v3.1 'Ankara'  ->", lookup(index31, "Ankara"))
print("v3.1 'Ankara,' ->", lookup(index31, "Ankara,"))   # must be None now

TEST PASSED - 8 words processed correctly.
TEST PASSED - 10 words processed correctly.
TEST PASSED - 5 words processed correctly.
Number of keywords in index31 (v3.1): 307
Number of keywords in index3  (v3.0): 350
TEST PASSED - 307 keywords checked, none contain punctuation.
TEST FAILED - 87 keyword(s) still contain punctuation.
Examples: ['project.', 'subpages.', 'tools.', 'regions.', 'recommendations.']
TEST PASSED - 307 keywords checked, no duplicates found.
keywords v3.0: 350  v3.1: 307
v3.0 'Ankara'  -> ['http://www.searchengineplaces.com.tr/travel_guide.html', 'http://www.searchengineplaces.com.tr/ankara.html']
v3.0 'Ankara,' -> ['http://www.searchengineplaces.com.tr/ankara.html']
v3.1 'Ankara'  -> ['http://www.searchengineplaces.com.tr/travel_guide.html', 'http://www.searchengineplaces.com.tr/ankara.html']
v3.1 'Ankara,' -> None


In [ ]:

#  QUESTION 3 - SEARCH ENGINE v3.2
#  v3.1 -> v3.2: keywords are converted to lowercase, so that
#                "Python" and "python" become one single entry.
#
#  No separate procedure is defined for the conversion: lower()
#  is applied directly inside addPageToIndex, right after the
#  punctuation has been removed.



def addPageToIndex_v32(index, url, content):
    """
    The v3.2 version of addPageToIndex. Two things happen to every word
    before it reaches the index:
      - noPunct removes the punctuation (inherited from v3.1)
      - lower() converts the result to lowercase (the v3.2 change)
    Lowercase was chosen as the single consistent form; upper() would
    have worked equally well, what matters is using one of them
    everywhere.
    """
    words = content.split()
    for word in words:
        keyword = noPunct(word).lower()      # v3.2: clean, then lowercase
        # A "word" made only of punctuation becomes an empty string after
        # cleaning. It is not a real word, so it is not indexed.
        if keyword != "":
            addToIndex(index, keyword, url)  # addToIndex from v3.0 is reused


def crawlWeb_v32(seed):
    """
    The v3.2 crawler. Structurally identical to v3.1; the only change is
    that it calls addPageToIndex_v32. The duplicate check of v3.0 still
    applies, since addToIndex is unchanged.
    """
    tocrawl = [seed]
    crawled = []
    index = {}
    while tocrawl:
        page = tocrawl.pop()
        if page not in crawled:
            content = getPage(page)
            # links must still be taken from the RAW html, before the tags
            # are removed by getClearPage
            union(tocrawl, get_all_links(content))
            addPageToIndex_v32(index, page, getClearPage(content))
            crawled.append(page)
    return index


# The seed page is crawled once more, now with the case conversion in
# place. index3 and index31 are left untouched and stay comparable.
seed = "https://searchengineplaces.com.tr/"
index3_2 = crawlWeb_v32(seed)

print("Number of keywords in index3_2 (v3.2):", len(index3_2))
print("Number of keywords in index31  (v3.1):", len(index31))
print("Number of keywords in index3   (v3.0):", len(index3))


# ---------- Testing v3.2 ----------

def testLowerCase(index):
    """
    Goes through every keyword of the index and compares it with its own
    lowercase form. If the two differ, that keyword still contains at
    least one uppercase letter, which means the conversion did not reach
    it. Comparing a string with keyword.lower() is enough, so we do not
    have to inspect the letters one by one.

    Only the failing keywords are collected, so the output stays short
    however large the index is.
    """
    problems = []
    for keyword in index:
        if keyword != keyword.lower():
            problems.append(keyword)

    if problems:
        print("TEST FAILED -", len(problems), "keyword(s) still contain uppercase letters.")
        print("Examples:", problems[:5])
        return False
    else:
        print("TEST PASSED -", len(index), "keywords checked, all are lowercase.")
        return True


# Test 1: the v3.2 index must be fully lowercase
testLowerCase(index3_2)

# Test 2: the same test on index31, the v3.1 index, which must FAIL.
# It still contains capitalised words such as "Ankara", so this proves
# that the test really detects uppercase letters instead of printing
# PASSED for whatever input it is given.
testLowerCase(index31)

# Test 3 and 4: the properties gained in the earlier versions must
# survive in v3.2 as well, so both earlier tests are run once more.
testIndexNoPunct(index3_2)     # no punctuation in the keywords (v3.1)
testNoDuplicates(index3_2)     # no duplicate URLs per keyword (v3.0)


# ---------- Comparing the versions ----------

# Each version merges more variants into a single keyword, so the number
# of keywords must decrease at every step: v3.0 > v3.1 > v3.2.
print("keywords  v3.0:", len(index3),
      " v3.1:", len(index31),
      " v3.2:", len(index3_2))

# A concrete before/after comparison on a single word.
print("v3.1 'Ankara' ->", lookup(index31, "Ankara"))
print("v3.1 'ankara' ->", lookup(index31, "ankara"))
print("v3.2 'Ankara' ->", lookup(index3_2, "Ankara"))   # must be None now
print("v3.2 'ankara' ->", lookup(index3_2, "ankara"))

Number of keywords in index3_2 (v3.2): 286
Number of keywords in index31  (v3.1): 307
Number of keywords in index3   (v3.0): 350
TEST PASSED - 286 keywords checked, all are lowercase.
TEST FAILED - 72 keyword(s) still contain uppercase letters.
Examples: ['Welcome', 'This', 'Our', 'Travel', 'Guide']
TEST PASSED - 286 keywords checked, none contain punctuation.
TEST PASSED - 286 keywords checked, no duplicates found.
keywords  v3.0: 350  v3.1: 307  v3.2: 286
v3.1 'Ankara' -> ['http://www.searchengineplaces.com.tr/travel_guide.html', 'http://www.searchengineplaces.com.tr/ankara.html']
v3.1 'ankara' -> None
v3.2 'Ankara' -> None
v3.2 'ankara' -> ['http://www.searchengineplaces.com.tr/travel_guide.html', 'http://www.searchengineplaces.com.tr/ankara.html']


In [ ]:

#  QUESTION 4 - PAGE RANKING



# ---------- Part a: crawling into an index and a graph ----------


def crawlWeb_v32g(seed):
    """
   What the graph represents: it is a dictioanry. Its keys are the crawled pages,  and each value
   is the list of URLs that page links to. The graph represents the link structure of the crawled web.
   And it tells us which pages point to a page. Ranking needs the graphbecause a page is popular when many popular pages link to it.
    """
    tocrawl = [seed]
    crawled = []
    graph = {}                 # url -> its outlinks
    index = {}
    while tocrawl:
        page = tocrawl.pop()
        if page not in crawled:
            content = getPage(page)
            outlinks = get_all_links(content)   # from the RAW html
            graph[page] = outlinks
            union(tocrawl, outlinks)
            addPageToIndex_v32(index, page, getClearPage(content))
            crawled.append(page)
    return index, graph


# two return values, unpacked into two variables
index3_2, graph1 = crawlWeb_v32g(seed)


 # ---------- Part b:printing the graph----------



def graphToString(graph):
    """
    Builds the printable text and returns it. "\\n" breaks the line, "\\t" indents.
    """
    result = "The graph has " + str(len(graph)) + " elements. These are :\n"
    counter = 1
    for page in graph:
        result = result + "\t" + str(counter) + " " + page + ":" + str(graph[page]) + "\n"
        counter = counter + 1
    return result

print(graphToString(graph1))


The graph has 10 elements. These are :
	1 https://searchengineplaces.com.tr/:['http://www.searchengineplaces.com.tr/travel_guide.html']
	2 http://www.searchengineplaces.com.tr/travel_guide.html:['http://www.searchengineplaces.com.tr/ankara.html', 'http://www.searchengineplaces.com.tr/konya.html', 'http://www.searchengineplaces.com.tr/istanbul.html', 'http://www.searchengineplaces.com.tr/oktayrecommends.html', 'http://www.searchengineplaces.com.tr/seymarecommends.html']
	3 http://www.searchengineplaces.com.tr/seymarecommends.html:['http://www.searchengineplaces.com.tr/oktayrecommends.html', 'http://www.searchengineplaces.com.tr/konya.html']
	4 http://www.searchengineplaces.com.tr/oktayrecommends.html:['http://www.searchengineplaces.com.tr/istanbul.html']
	5 http://www.searchengineplaces.com.tr/istanbul.html:['http://www.searchengineplaces.com.tr/maidens_tower.html', 'http://www.searchengineplaces.com.tr/galata_tower.html']
	6 http://www.searchengineplaces.com.tr/galata_tower.html:['http

In [ ]:
# ---------- Part c: computing the ranks ----------


def computeRanks(graph):

  d = 0.8 #damping factor
  N = len(graph) #number of pages
  numloops = 10 #effects the accuracy
  ranks = {}
  for page in graph:
      ranks[page] = 1/N

  for i in range(0, numloops):
       newranks = {}
       for page in graph:
           newrank = (1-d)/N
           for node in graph:
              if page in graph[node]:
                   newrank = newrank + d * (ranks[node]/len(graph[node]))
           newranks[page] = newrank

       ranks = newranks
  return newranks

def ranksToString(ranks):
    result= ""
    for page in ranks:
      result= result+ "The rank of the page" + page + ":" +str(ranks[page])+ "\n"
    return result

ranks1= computeRanks(graph1)
print(ranksToString(ranks1))

The rank of the pagehttps://searchengineplaces.com.tr/:0.019999999999999997
The rank of the pagehttp://www.searchengineplaces.com.tr/travel_guide.html:0.14780429869056003
The rank of the pagehttp://www.searchengineplaces.com.tr/seymarecommends.html:0.073060618584064
The rank of the pagehttp://www.searchengineplaces.com.tr/oktayrecommends.html:0.073060618584064
The rank of the pagehttp://www.searchengineplaces.com.tr/istanbul.html:0.17460832634470402
The rank of the pagehttp://www.searchengineplaces.com.tr/galata_tower.html:0.09025810358272002
The rank of the pagehttp://www.searchengineplaces.com.tr/maidens_tower.html:0.09025810358272002
The rank of the pagehttp://www.searchengineplaces.com.tr/konya.html:0.073060618584064
The rank of the pagehttp://www.searchengineplaces.com.tr/mevlana.html:0.04928445079552
The rank of the pagehttp://www.searchengineplaces.com.tr/ankara.html:0.043776167788544



In [ ]:
# ---------- Part d: lookUp  ----------

def rankedLookup(index, key, graph):
  """
  Like lookup, but sorted by rank, highest first,sorted() returns a new list, so the index
  is not reordered
  """

  if key not in index:
    return None

  ranks= computeRanks(graph)
  return sorted(index[key], key= ranks.get, reverse= True)

In [ ]:
results= rankedLookup(index3_2, "istanbul", graph1)
for result in results:
  print(result)

http://www.searchengineplaces.com.tr/istanbul.html
http://www.searchengineplaces.com.tr/travel_guide.html
http://www.searchengineplaces.com.tr/galata_tower.html
http://www.searchengineplaces.com.tr/maidens_tower.html
http://www.searchengineplaces.com.tr/oktayrecommends.html
